# PQ-SetFormer — one-pass P100 training

Run this in the `filament-segmentation-2026` Kaggle competition notebook environment with a P100 GPU and Internet enabled. It uses Kaggle’s resident CUDA driver and toolkit. Kaggle’s current PyTorch wheel excludes the P100’s Pascal `sm_60` architecture, so the notebook installs a compatible CUDA 12.6 PyTorch wheel before any PyTorch import, then builds Detectron2 and the MaskDINO P100 operator against it. It trains once over all annotated train images, writes JSONL and offline W&B logs, then prints exact per-image training-monitor PQ. The test mount is validated but never used for supervised training or monitor PQ. Save a Kaggle version to retain `/kaggle/working/pq-setformer`.

In [ ]:
from pathlib import Path
COMPETITION_ROOT = Path('/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026')
TRAIN_ROOT = COMPETITION_ROOT / 'train'
TEST_ROOT = COMPETITION_ROOT / 'test'
INPUT_SIZE = 1024
TOTAL_MICROBATCHES = 6_000
MONITOR_IMAGES = 32
assert (TRAIN_ROOT / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json').is_file(), f'Missing competition annotations: {TRAIN_ROOT}'
assert (TRAIN_ROOT / 'train_images').is_dir(), f'Missing competition train images: {TRAIN_ROOT}'
assert TEST_ROOT.is_dir(), f'Missing competition test mount: {TEST_ROOT}'
assert INPUT_SIZE >= 1024 and INPUT_SIZE % 32 == 0 and TOTAL_MICROBATCHES % 4 == 0
print({'train_root': str(TRAIN_ROOT), 'test_root': str(TEST_ROOT), 'input_size': INPUT_SIZE, 'microbatches': TOTAL_MICROBATCHES,
       'optimizer_updates': TOTAL_MICROBATCHES // 4})

In [ ]:
PAYLOAD = '...'
import base64, json, shutil, zlib
from pathlib import Path
WORK = Path('/kaggle/working')
RUN_ROOT = WORK / 'pq-setformer'
if RUN_ROOT.exists():
    shutil.rmtree(RUN_ROOT)
RUN_ROOT.mkdir()
SOURCES = json.loads(zlib.decompress(base64.b64decode(PAYLOAD)))
for name, text in SOURCES.items():
    path = WORK / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
print('Unpacked', len(SOURCES), 'frozen source files')



In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

if "torch" in sys.modules:
    raise RuntimeError(
        "PyTorch is already imported in this kernel. Restart the Kaggle session, "
        "then Run All from the first cell so the P100-compatible PyTorch wheel can load."
    )

os.environ.update(TORCH_CUDA_ARCH_LIST="6.0", WANDB_MODE="offline", MAX_JOBS="2")


def run(*args, cwd=None, quiet=False):
    if not quiet:
        subprocess.check_call(list(args), cwd=cwd)
        return
    result = subprocess.run(
        list(args), cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    if result.returncode:
        print(result.stdout[-12_000:])
        raise subprocess.CalledProcessError(result.returncode, list(args), output=result.stdout)


# Kaggle's current PyTorch wheel drops Pascal sm_60. This installs a P100-capable
# PyTorch binary; it does not install or replace the CUDA driver/toolkit.
run(
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--force-reinstall",
    # PyTorch CUDA wheels require their matching NVIDIA runtime packages (including cuSPARSELt).
    "torch==2.7.1",
    "torchvision==0.22.1",
    "--index-url",
    "https://download.pytorch.org/whl/cu126",
)
import torch

assert torch.cuda.is_available(), "Enable a Kaggle P100 GPU accelerator before Run All"
name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
architectures = torch.cuda.get_arch_list()
assert "P100" in name and capability == (6, 0), (name, capability)
assert "sm_60" in architectures, f"P100-capable PyTorch installation failed: {architectures}"
assert shutil.which("nvcc"), "The Kaggle CUDA toolkit is required to compile MaskDINO"
print({"torch": torch.__version__, "torch_cuda": torch.version.cuda, "gpu": name, "capability": capability})

# Project-level packages and source-built extensions.
run(sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--no-deps", "wandb==0.30.0", "pycocotools==2.0.11", "scipy==1.15.3", "timm==0.9.16", "ninja==1.11.1.4", "tqdm==4.67.1")
run(sys.executable, "-m", "pip", "install", "--quiet", "--no-build-isolation", "git+https://github.com/facebookresearch/detectron2.git@a2f4a8771ab77e8411c26b27f24f9489a28a2453")
vendor = WORK / "vendor"
run("git", "clone", "--quiet", "https://github.com/IDEA-Research/MaskDINO.git", str(vendor))
run("git", "checkout", "--quiet", "3831d8514a3728535ace8d4ecc7d28044c42dd14", cwd=vendor)
path = vendor / "maskdino/modeling/transformer_decoder/maskdino_decoder.py"
text = path.read_text(); anchor = "        out = {\n            'pred_logits': predictions_class[-1],"
replacement = "        # PQ quality adapter\n        pq_query_features = hs[-1]\n        if mask_dict is not None:\n            pq_query_features = pq_query_features[:, mask_dict['pad_size']:, :]\n        out = {\n            'pred_logits': predictions_class[-1],"
assert text.count(anchor) == 1; text = text.replace(anchor, replacement)
anchor = "            'pred_boxes':out_boxes[-1],"; replacement = "            'pred_boxes':out_boxes[-1],\n            'pred_query_features': pq_query_features,\n            'pixel_features': mask_features,"
assert text.count(anchor) == 1; path.write_text(text.replace(anchor, replacement))
path = vendor / "maskdino/modeling/pixel_decoder/ops/src/cuda/ms_deform_attn_cuda.cu"; text = path.read_text(); anchor = "AT_DISPATCH_FLOATING_TYPES(value.type(),"
assert text.count(anchor) == 2; path.write_text(text.replace(anchor, "AT_DISPATCH_FLOATING_TYPES(value.scalar_type(),"))
run("sh", "make.sh", cwd=vendor / "maskdino/modeling/pixel_decoder/ops", quiet=True)
print("Kaggle P100 runtime and MaskDINO extension are ready")


In [ ]:
import hashlib, json, os, runpy, sys
from dataclasses import replace
from datetime import UTC, datetime
sys.path[:0] = [str(WORK), str(WORK / 'vendor')]
os.chdir(WORK)
import config, data
root = data.discover_dataset_root(TRAIN_ROOT)
if root != TRAIN_ROOT.resolve():
    raise RuntimeError(f'Competition training root drifted: {root}')
index = data.load_annotation_index(root / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json')
training_names = list(index.filenames)
stride = max(1, len(training_names) // MONITOR_IMAGES)
inference_names = training_names[::stride][:MONITOR_IMAGES]
experiment = config.ExperimentConfig(model=replace(config.ModelConfig(), input_size=INPUT_SIZE), training=replace(config.TrainingConfig(), target_microbatches=TOTAL_MICROBATCHES))
def digest(value): return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
run_id = 'full-' + datetime.now(UTC).strftime('%Y%m%d-%H%M%S')
membership = {'training_names': training_names, 'inference_names': inference_names, 'role': 'full-fit-training-monitor'}
task = {'run_id': run_id, 'stage': f'full-train-{INPUT_SIZE}', 'run_kind': 'pilot', 'training_microbatches': TOTAL_MICROBATCHES, 'source_sha256': digest(SOURCES), 'source_files': list(SOURCES), 'vendor_lock_sha256': hashlib.sha256(SOURCES['vendor-lock.json'].encode()).hexdigest(), 'git_sha': 'notebook-frozen-' + digest(SOURCES)[:24], 'membership_sha256': digest(membership), 'config_sha256': experiment.sha256, 'experiment': experiment.to_dict(), 'training_names': training_names, 'inference_names': inference_names, 'split': membership, 'dataset': {'source': 'kaggle-competition/filament-segmentation-2026', **data.training_dataset_snapshot(root)}}
(WORK / 'task-config.json').write_text(json.dumps(task, sort_keys=True))
print({'run_id': run_id, 'training_images': len(training_names), 'monitor_images': len(inference_names)})
runpy.run_path(str(WORK / 'remote.py'), run_name='__main__')

In [ ]:
import json
import pandas as pd
report = json.loads((RUN_ROOT / 'pilot-report.json').read_text())
print(json.dumps({'status': report['status'], 'global_pq': report['inference']['pq'], 'sq': report['inference']['sq'], 'rq': report['inference']['rq'], 'optimizer_steps': report['training']['optimizer_steps'], 'peak_reserved_gb': report['training']['peak_reserved_gb'], 'checkpoint_sha256': report['checkpoint_sha256']}, indent=2))
display(pd.DataFrame.from_dict(report['inference']['files'], orient='index').rename_axis('filename').reset_index().sort_values('pq', ascending=False))
print('Artifacts:', RUN_ROOT)